# 04. Dataset teks khusus dan prapemrosesan

Membaca CSV, memeriksa kualitas data, membangun kosakata latih, mengukur OOV, dan menyiapkan batch dinamis.

**Prasyarat:** modul 02.

**Pola belajar:** baca penjelasan, prediksi bentuk keluaran, jalankan kode, lalu ubah satu hal.

Contoh ulasan dalam paket ini merupakan data sintetis untuk mempelajari mekanisme. Metriknya tidak mewakili kinerja pada ulasan nyata.

## Penyiapan

Instal dependensi melalui petunjuk README sebelum menjalankan seluruh sel. Setiap notebook dapat dimulai dengan kernel baru. GPU bersifat opsional. Semua operasi tensor yang berinteraksi harus berada pada perangkat yang sesuai.

In [1]:
from pathlib import Path
import sys
# Lokal: buka dari root repo, folder nlp, atau nlp/notebooks.
# Colab: ambil paket kursus jika belum tersedia.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for base in candidates for p in (base, base / "nlp")
             if (p / "nlp_course").is_dir()), None)
if ROOT is None and "google.colab" in sys.modules:
    import subprocess
    target = Path("/content/pytorch-deep-learning-nlp")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                        "--sparse", "--branch", "nlp-learning-path",
                        "https://github.com/FeliksMakarios/pytorch-deep-learning.git",
                        str(target)], check=True)
        subprocess.run(["git", "sparse-checkout", "set", "nlp"], cwd=target, check=True)
    ROOT = target / "nlp"
if ROOT is None or not (ROOT / "nlp_course").is_dir():
    raise RuntimeError("Folder nlp_course tidak ditemukan. Ikuti petunjuk README nlp.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import torch
from torch import nn
from nlp_course.data import tokenize, build_vocab, encode, read_rows, loaders, collate_batch
from nlp_course.models import MeanClassifier, RecurrentClassifier, TinyTransformer
from nlp_course.engine import seed_all, fit, run_epoch, metrics, save_mean, load_mean, predict
seed_all(42)
torch.set_num_threads(1)
device = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
print("PyTorch:", torch.__version__, "Perangkat:", device)

PyTorch: 2.14.0+cu130 Perangkat: cpu


## 1. Kontrak dataset

CSV inti memiliki kolom text, label, split. Kolom domain dan template_id membantu audit data sintetis. Untuk data nyata, pembagian berdasarkan penulis, percakapan, produk, atau waktu mungkin lebih tepat daripada acak per baris.

In [2]:
import csv
from collections import Counter
rows = read_rows()
print(rows[0])
print(Counter(r["split"] for r in rows))
print(Counter(r["domain"] for r in rows))

{'text': 'buku ini buruk', 'label': 0, 'split': 'train', 'domain': 'produk', 'template_id': 'train_0'}
Counter({'train': 288, 'val': 96, 'test': 96})
Counter({'produk': 240, 'layanan': 240})


## 2. Audit duplikasi dan kelompok

Duplikat persis antarsplit dapat membuat evaluasi terlalu optimistis. Pemeriksaan berikut juga memastikan kelompok template tidak berpindah split. Data nyata memerlukan pemeriksaan duplikat dekat yang lebih lanjut.

In [3]:
parts = {s:[r for r in rows if r["split"]==s] for s in ["train","val","test"]}
sets = {s:{" ".join(tokenize(r["text"])) for r in part} for s,part in parts.items()}
groups = {s:{r["template_id"] for r in part} for s,part in parts.items()}
for a,b in [("train","val"),("train","test"),("val","test")]:
    assert sets[a].isdisjoint(sets[b])
    assert groups[a].isdisjoint(groups[b])
print("Tidak ada duplikat atau kelompok template yang melintasi split.")

Tidak ada duplikat atau kelompok template yang melintasi split.


## 3. Normalisasi tidak selalu berarti menghapus

Tokenisasi sederhana menggunakan huruf kecil. Keputusan ini sesuai demonstrasi sentimen, tetapi dapat menghilangkan informasi nama entitas. Hindari penghapusan negasi. Simpan teks asli untuk pelacakan kesalahan.

In [4]:
for text in ["TIDAK bagus!", "Saya membeli Buku.", "", "baik 😊"]:
    print(repr(text),tokenize(text))

'TIDAK bagus!' ['tidak', 'bagus', '!']
'Saya membeli Buku.' ['saya', 'membeli', 'buku', '.']
'' []
'baik 😊' ['baik', '😊']


## 4. Mengukur kata di luar kosakata

OOV adalah token yang tidak terdapat dalam kosakata latih. Rasio tinggi dapat menandai pergeseran domain. Memasukkan kata uji ke kosakata untuk menurunkan rasio akan mengubah protokol evaluasi.

In [5]:
vocab = build_vocab(r["text"] for r in parts["train"])
for split,part in parts.items():
    tokens = [t for r in part for t in tokenize(r["text"])]
    oov = sum(t not in vocab for t in tokens)
    print(split,"OOV:",round(oov/len(tokens),3))
print("Teks kosong menjadi:",encode("",vocab))

train OOV: 0.0
val OOV: 0.462
test OOV: 0.5
Teks kosong menjadi: [1]


## 5. Panjang maksimum dan pemotongan

Pilih batas panjang berdasarkan distribusi data latih. Pemotongan dapat membuang negasi atau kesimpulan di akhir kalimat. Laporkan proporsi teks yang terpotong. Model Transformer memiliki kapasitas posisi yang harus selaras.

In [6]:
lengths = torch.tensor([len(tokenize(r["text"])) for r in parts["train"]],dtype=torch.float32)
print("Median, persentil 95:",torch.quantile(lengths,torch.tensor([0.5,0.95])))
max_length = 8
print("Proporsi latih terpotong:",(lengths>max_length).float().mean().item())
print(encode("buku ini sangat baik tetapi akhir ceritanya tidak memuaskan",vocab,max_length))

Median, persentil 95: tensor([5., 6.])
Proporsi latih terpotong: 0.0
[4, 9, 20, 3, 1, 1, 1, 25]


## 6. Dataset dan collate_fn

Dataset mengembalikan satu contoh. collate_fn menyatukan contoh yang panjangnya berbeda. Padding dinamis hanya memanjang hingga urutan terpanjang dalam batch sehingga mengurangi komputasi yang tidak diperlukan.

In [7]:
from nlp_course.data import TextDataset
from torch.utils.data import DataLoader
dataset = TextDataset(parts["train"],vocab,max_length=64)
loader = DataLoader(dataset,batch_size=5,collate_fn=collate_batch)
ids,lengths,labels = next(iter(loader))
print(ids.shape,lengths,labels)
assert ids.dtype == labels.dtype == torch.long
assert torch.equal(ids.ne(0).sum(1),lengths)

torch.Size([5, 4]) tensor([3, 3, 4, 4, 3]) tensor([0, 0, 0, 0, 1])


## 7. Memasukkan data milik Anda

Siapkan CSV dengan label 0=negatif dan 1=positif. Tentukan split sebelum menjalankan loaders. Sel berikut menguji jalur CSV menggunakan salinan data demonstrasi. Ganti path dengan CSV nyata setelah audit. Kursus ini menyediakan notebook tambahan untuk dataset publik SST-2.

In [8]:
custom_path = ARTIFACTS / "contoh_format.csv"
with custom_path.open("w",newline="",encoding="utf-8") as f:
    writer = csv.DictWriter(f,fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
custom_rows = read_rows(custom_path)
custom_vocab,custom_train,custom_val,custom_test = loaders(custom_rows)
print("CSV berhasil dimuat:",len(custom_rows))

CSV berhasil dimuat: 480


## Latihan mandiri

1. Apa risiko membagi dua ulasan dari pengguna yang sama secara acak?
2. Kapan lowercasing merugikan?
3. Mengapa teks kosong dipetakan ke UNK untuk latihan namun ditolak aplikasi?
4. Bagaimana memilih panjang maksimum?

## Pembahasan latihan

1. Model dapat mengenali ciri pengguna yang muncul pada kedua split. Gunakan pembagian kelompok jika relevan.
2. Saat kapitalisasi membantu membedakan nama orang, organisasi, atau akronim.
3. UNK mencegah urutan nol pada pipeline teknis. Aplikasi meminta masukan bermakna dari pengguna.
4. Tinjau distribusi latih, biaya komputasi, dan dampak pemotongan pada validasi.

## Penghubung ke materi berikutnya

Modul 05 memisahkan kode data, model, pelatihan, dan prediksi agar eksperimen lebih mudah diulang.

### Rujukan
- [Dokumentasi PyTorch](https://docs.pytorch.org/docs/stable/index.html)
- [Sumber Embedding](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/sparse.py)
- [Sumber Transformer](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/transformer.py)
- [Kursus sumber dan struktur awal](https://github.com/mrdbourke/pytorch-deep-learning)

Materi ini ditulis sebagai jalur NLP mandiri. Penjelasan dan contoh NLP bukan terjemahan resmi kursus sumber.